In [1]:
import numpy as np
import pandas as pd


In [2]:
from pathlib import Path
import os

root = Path.cwd().parent
os.chdir(root)

In [3]:
df = pd.read_csv(r'data\train.csv')

In [4]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [5]:
df.dropna(inplace=True)
df.drop_duplicates(inplace = True)

In [6]:
df.shape

(404287, 6)

In [7]:
ques_df = df[['question1', 'question2']]

In [8]:
ques_df

,question1,question2
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?
...,...,...
404285,How many keywords are there in the Racket prog...,How many keywords are there in PERL Programmin...
404286,Do you believe there is life after death?,Is it true that there is life after death?
404287,What is one coin?,What's this coin?
404288,What is the approx annual cost of living while...,I am having little hairfall problem but I want...


### applying bow only

In [9]:
# applying bow on the list

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features = 3000)
ques1_vec = vectorizer.fit_transform(ques_df['question1'])
ques2_vec = vectorizer.fit_transform(ques_df['question2'])


In [10]:
print(ques1_vec.shape)
print(ques2_vec.shape)

(404287, 3000)
(404287, 3000)


In [12]:
# horizontally stacking the 2 vecators

from scipy.sparse import hstack

X = hstack([ques1_vec, ques2_vec])

In [13]:
X.shape

(404287, 6000)

In [14]:
y = df['is_duplicate'].values

In [15]:
y.shape

(404287,)

### Applying different algorithms

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [17]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [18]:
print(X_train.shape)
print(y_test.shape)

(323429, 6000)
(80858,)


In [ ]:
# rf = RandomForestClassifier()
# rf.fit(X_train, y_train)
# y_pred = rf.predict(X_test)
# print(accuracy_score(y_test, y_pred))

In [ ]:
# import pickle

# os.makedirs('artifacts', exist_ok=True)

# with open('artifacts/base_model.pkl', 'wb') as file:
#     pickle.dump(rf, file)

In [19]:
xgb = XGBClassifier()
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.7477429567884439


In [20]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.7369091493729748


c:\Users\apaks\anaconda3\envs\quora-myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [22]:
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB()
mnb.fit(X_train, y_train)
y_pred = mnb.predict(X_test)
print('MNB score: ', accuracy_score(y_test, y_pred))

MNB score:  0.7145489623784906
